# CRYPTO SCALPER LAB v0.6 — Strategy Tournament
Öt stratégiafamilád, 7 USDC-piac, walk-forward development gate, observed diagnostic, lezárt holdout.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive csatlakoztatva.')


In [ ]:
from pathlib import Path
import shutil, subprocess
PROJECT=Path('/content/drive/MyDrive/CRYPTO_SCALPER_LAB_v04')
DATA=PROJECT/'data'
RUNS=PROJECT/'runs'
REPO=Path('/content/crypto-scalper-lab')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','https://github.com/bognar-aron/bognar-aron-crypto-scalper-lab.git',str(REPO)],check=True)
SRC=REPO
required=['strategy_tournament_v06.py','backtester.py','requirements.txt']
missing=[x for x in required if not (SRC/x).exists()]
print('GitHub forrás:',SRC)
print('Drive adatcache:',DATA)
print('Drive runok:',RUNS)
assert not missing, f'Hiányzó v0.6 fájlok: {missing}'


In [ ]:
!pip -q install -r '/content/crypto-scalper-lab/requirements.txt'


In [ ]:
SYMBOLS='SOLUSDC,BTCUSDC,ETHUSDC,BNBUSDC,XRPUSDC,ADAUSDC,DOGEUSDC'
STARTING_EQUITY=2500
MODE='fast'
FEE_BPS=10.0
SLIPPAGE_BPS=2.0
START='2024-01-01'
END='2026-07-31'
UNLOCK_HOLDOUT=False
print({'symbols':SYMBOLS,'mode':MODE,'start':START,'end':END,'unlock_holdout':UNLOCK_HOLDOUT})

In [ ]:
import subprocess,sys,datetime
stamp=datetime.datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUT=RUNS/f'v06_TOURNAMENT_{MODE}_{stamp}'
OUT.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,str(SRC/'strategy_tournament_v06.py'),'--symbols',SYMBOLS,'--equity',str(STARTING_EQUITY),'--mode',MODE,'--fee-bps',str(FEE_BPS),'--slippage-bps',str(SLIPPAGE_BPS),'--start',START,'--end',END,'--data-dir',str(DATA),'--out',str(OUT)]
if UNLOCK_HOLDOUT: cmd += ['--unlock-holdout','--holdout-end-exclusive','2026-09-01']
proc=subprocess.run(cmd,text=True,capture_output=True)
print(proc.stdout)
if proc.returncode!=0:
 print(proc.stderr); raise RuntimeError(f'v0.6 hibával állt le: {proc.returncode}')
print('Eredmények:',OUT)

In [ ]:
import json,pandas as pd
from IPython.display import display
display(pd.read_csv(OUT/'symbol_coverage.csv'))
gate=json.loads((OUT/'development_gate.json').read_text())
print('DEVELOPMENT GATE')
for k,v in gate.items(): print(k,':',v)

In [ ]:
leader=pd.read_csv(OUT/'tournament_leaderboard.csv')
display(leader)
print('Családonkénti bajnokok:')
display(pd.read_csv(OUT/'family_champions.csv'))

In [ ]:
print('Development cost sensitivity:')
display(pd.read_csv(OUT/'development_cost_sensitivity.csv'))

In [ ]:
obs=OUT/'observed_test_summary.json'
if obs.exists():
 s=json.loads(obs.read_text()); print('OBSERVED TEST'); [print(k,':',v) for k,v in s.items()]
else: print('Observed test nem futott: development FAIL.')
locked=OUT/'SEALED_HOLDOUT_LOCKED.txt'
if locked.exists(): print(locked.read_text())

In [ ]:
import shutil
archive=shutil.make_archive(str(OUT),'zip',OUT)
print('ZIP:',archive)